# 🧠 Sentiment Analysis — Exploratory Data Analysis & Model Training

This notebook covers:
1. **Dataset Exploration** — Loading and understanding the Sentiment140 dataset
2. **Text Preprocessing** — Cleaning and preparing tweets for analysis
3. **Exploratory Visualizations** — Word frequency, sentiment distribution, text length analysis
4. **Model Comparison** — VADER vs TextBlob vs DistilBERT accuracy
5. **Performance Metrics** — Confusion matrices and classification reports

## 📦 Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import re
import warnings
warnings.filterwarnings('ignore')

# NLP Libraries
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from textblob import TextBlob
import spacy

# Set style
plt.style.use('dark_background')
sns.set_palette('viridis')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

print('All imports successful! ✅')

## 📊 1. Load Dataset

We use the **Sentiment140 dataset** (1.6M tweets) from Kaggle.
- Download from: https://www.kaggle.com/datasets/kazanova/sentiment140
- Or use the included `sample_tweets.csv` for quick exploration.

In [ ]:
# Option 1: Load sample dataset
df = pd.read_csv('../data/sample_tweets.csv')

# Option 2: Load full Sentiment140 dataset (uncomment below)
# COLUMNS = ['target', 'ids', 'date', 'flag', 'user', 'text']
# df = pd.read_csv('../data/training.1600000.processed.noemoticon.csv',
#                   encoding='latin-1', names=COLUMNS)
# df['target'] = df['target'].map({0: 'negative', 2: 'neutral', 4: 'positive'})

print(f'Dataset shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head()

## 📈 2. Basic Statistics

In [ ]:
# Text length distribution
df['text_length'] = df['text'].astype(str).apply(len)
df['word_count'] = df['text'].astype(str).apply(lambda x: len(x.split()))

print('Text Length Statistics:')
print(df['text_length'].describe())
print(f'\nWord Count Statistics:')
print(df['word_count'].describe())

In [ ]:
# Visualize text length distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['text_length'], bins=50, color='#7c5cfc', edgecolor='#1a1a2e', alpha=0.8)
axes[0].set_title('Distribution of Text Length (Characters)', fontweight='bold')
axes[0].set_xlabel('Character Count')
axes[0].set_ylabel('Frequency')

axes[1].hist(df['word_count'], bins=30, color='#00d4ff', edgecolor='#1a1a2e', alpha=0.8)
axes[1].set_title('Distribution of Word Count', fontweight='bold')
axes[1].set_xlabel('Word Count')
axes[1].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

## 🧹 3. Text Preprocessing

In [ ]:
def clean_text(text):
    """Clean tweet text for analysis."""
    text = str(text)
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)  # Remove URLs
    text = re.sub(r'@\w+', '', text)                       # Remove mentions
    text = re.sub(r'#(\w+)', r'\1', text)                  # Remove # but keep word
    text = re.sub(r'\s+', ' ', text).strip()               # Clean whitespace
    return text

df['clean_text'] = df['text'].apply(clean_text)
df[['text', 'clean_text']].head(10)

## 🔬 4. VADER Sentiment Analysis

In [ ]:
nltk.download('vader_lexicon', quiet=True)
sia = SentimentIntensityAnalyzer()

def vader_sentiment(text):
    scores = sia.polarity_scores(str(text))
    if scores['compound'] >= 0.05:
        return 'positive'
    elif scores['compound'] <= -0.05:
        return 'negative'
    return 'neutral'

df['vader_sentiment'] = df['clean_text'].apply(vader_sentiment)
df['vader_score'] = df['clean_text'].apply(lambda x: sia.polarity_scores(str(x))['compound'])

print('VADER Sentiment Distribution:')
print(df['vader_sentiment'].value_counts())

## 🔬 5. TextBlob Sentiment Analysis

In [ ]:
def textblob_sentiment(text):
    polarity = TextBlob(str(text)).sentiment.polarity
    if polarity > 0.1:
        return 'positive'
    elif polarity < -0.1:
        return 'negative'
    return 'neutral'

df['textblob_sentiment'] = df['clean_text'].apply(textblob_sentiment)
df['textblob_score'] = df['clean_text'].apply(lambda x: TextBlob(str(x)).sentiment.polarity)

print('TextBlob Sentiment Distribution:')
print(df['textblob_sentiment'].value_counts())

## 📊 6. Comparison Visualizations

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

colors = ['#00e676', '#ff5252', '#ffc107']
order = ['positive', 'negative', 'neutral']

# VADER distribution
vader_counts = df['vader_sentiment'].value_counts().reindex(order)
axes[0].bar(order, vader_counts.values, color=colors, edgecolor='#1a1a2e', linewidth=1.5)
axes[0].set_title('VADER Sentiment Distribution', fontweight='bold', fontsize=14)
axes[0].set_ylabel('Count')
for i, v in enumerate(vader_counts.values):
    axes[0].text(i, v + 1, str(v), ha='center', fontweight='bold', color='white')

# TextBlob distribution
tb_counts = df['textblob_sentiment'].value_counts().reindex(order)
axes[1].bar(order, tb_counts.values, color=colors, edgecolor='#1a1a2e', linewidth=1.5)
axes[1].set_title('TextBlob Sentiment Distribution', fontweight='bold', fontsize=14)
axes[1].set_ylabel('Count')
for i, v in enumerate(tb_counts.values):
    axes[1].text(i, v + 1, str(v), ha='center', fontweight='bold', color='white')

plt.tight_layout()
plt.show()

In [ ]:
# Score comparison scatter plot
fig, ax = plt.subplots(figsize=(10, 8))

scatter = ax.scatter(
    df['vader_score'], df['textblob_score'],
    c=df['vader_sentiment'].map({'positive': '#00e676', 'negative': '#ff5252', 'neutral': '#ffc107'}),
    alpha=0.6, edgecolors='#1a1a2e', linewidth=0.5, s=50
)

ax.set_xlabel('VADER Compound Score', fontsize=12)
ax.set_ylabel('TextBlob Polarity Score', fontsize=12)
ax.set_title('VADER vs TextBlob Score Comparison', fontweight='bold', fontsize=14)
ax.axhline(y=0, color='white', linestyle='--', alpha=0.3)
ax.axvline(x=0, color='white', linestyle='--', alpha=0.3)

# Legend
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#00e676', markersize=10, label='Positive'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#ff5252', markersize=10, label='Negative'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#ffc107', markersize=10, label='Neutral'),
]
ax.legend(handles=legend_elements, loc='lower right')

plt.tight_layout()
plt.show()

## 🌐 7. Most Common Words per Sentiment

In [ ]:
from nltk.corpus import stopwords
nltk.download('stopwords', quiet=True)

stop_words = set(stopwords.words('english'))

def get_top_words(sentiment, n=20):
    texts = df[df['vader_sentiment'] == sentiment]['clean_text'].str.lower()
    all_words = ' '.join(texts).split()
    filtered = [w for w in all_words if w not in stop_words and len(w) > 2 and w.isalpha()]
    return Counter(filtered).most_common(n)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for idx, (sentiment, color) in enumerate([('positive', '#00e676'), ('negative', '#ff5252'), ('neutral', '#ffc107')]):
    top = get_top_words(sentiment, 15)
    if top:
        words, counts = zip(*top)
        axes[idx].barh(range(len(words)), counts, color=color, alpha=0.8)
        axes[idx].set_yticks(range(len(words)))
        axes[idx].set_yticklabels(words)
        axes[idx].invert_yaxis()
    axes[idx].set_title(f'Top Words — {sentiment.title()}', fontweight='bold')

plt.tight_layout()
plt.show()

## 🤖 8. DistilBERT Analysis (Optional — Requires GPU for Large Datasets)

This section uses HuggingFace Transformers to run DistilBERT on a subset of the data.
**Note:** This requires `transformers` and `torch` to be installed.

In [ ]:
try:
    from transformers import pipeline
    
    bert_pipeline = pipeline('sentiment-analysis',
                             model='distilbert-base-uncased-finetuned-sst-2-english',
                             truncation=True, max_length=512)
    
    # Analyze a subset (BERT is slow on CPU)
    sample_size = min(100, len(df))
    sample_df = df.head(sample_size).copy()
    
    bert_results = bert_pipeline(sample_df['clean_text'].tolist(), batch_size=16)
    
    sample_df['bert_label'] = [r['label'] for r in bert_results]
    sample_df['bert_score'] = [r['score'] for r in bert_results]
    sample_df['bert_sentiment'] = sample_df['bert_label'].map({
        'POSITIVE': 'positive', 'NEGATIVE': 'negative'
    })
    
    print('DistilBERT Sentiment Distribution (Sample):')
    print(sample_df['bert_sentiment'].value_counts())
    
except ImportError:
    print('⚠️ transformers not installed. Install with: pip install transformers torch')
except Exception as e:
    print(f'⚠️ BERT analysis failed: {e}')

## 📊 9. Model Agreement Analysis

In [ ]:
# Check agreement between VADER and TextBlob
agreement = (df['vader_sentiment'] == df['textblob_sentiment']).mean() * 100

print(f'VADER vs TextBlob Agreement: {agreement:.1f}%')

# Confusion-style heatmap
from sklearn.metrics import confusion_matrix

try:
    labels = ['positive', 'negative', 'neutral']
    cm = confusion_matrix(df['vader_sentiment'], df['textblob_sentiment'], labels=labels)
    
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='viridis',
                xticklabels=labels, yticklabels=labels, ax=ax)
    ax.set_title('VADER vs TextBlob — Confusion Matrix', fontweight='bold')
    ax.set_xlabel('TextBlob Prediction')
    ax.set_ylabel('VADER Prediction')
    plt.tight_layout()
    plt.show()
except ImportError:
    print('Install scikit-learn for confusion matrix: pip install scikit-learn')

## 📝 10. Summary & Conclusions

### Key Findings:

1. **VADER** is fast and works well with social media text (emojis, slang, punctuation)
2. **TextBlob** provides subjectivity scores but can be less accurate on informal text
3. **DistilBERT** offers the highest accuracy but requires significantly more compute
4. Model agreement varies — combining multiple models provides more robust results

### Recommendations:
- Use **VADER** as the primary model for social media analysis
- Use **DistilBERT** for formal/complex text where accuracy is critical
- The **model comparison view** in the web app helps users understand differences